# NB09 — Meth3D-Net V6: Complete Multi-Cancer + Lung Cancer Validation
## Google Colab Version · 7 Cancer Cohorts · Single Notebook
### Lung Cancer + Breast + Colorectal (×2) + HCC + GBM

---

> **Colab setup:** This notebook mounts your Google Drive and reads GEO series_matrix files
> stored in `MyDrive/Meth3DNet_V6/MultiCancer/`. V6 DMB files are downloaded from
> Zenodo (https://zenodo.org/records/19657976) automatically.

| Key | GEO | Cancer | n | Normal? | Layer C? |
|-----|-----|--------|---|---------|----------|
| `Lung_GSE39279` | GSE39279 | Lung adenocarcinoma | ~246 | ✓ Yes | ✓ Yes |
| `Lung_GSE56044` | GSE56044 | Lung cancer (multi-hist.) | ~119 | ✓ Yes | ✓ Yes |
| `Breast_GSE75067` | GSE75067 | Breast cancer | 188 | ✗ Tumour-only | ✗ No |
| `CRC_GSE101764` | GSE101764 | Colorectal cancer | ~280 | ✓ Yes | ✓ Yes |
| `CRC_GSE48684` | GSE48684 | Colorectal (progression) | ~147 | ✓ Yes | ✓ Yes |
| `HCC_GSE54503` | GSE54503 | Hepatocellular carcinoma | 47 | ✓ Yes | ✓ Yes |
| `GBM_GSE36278` | GSE36278 | Glioblastoma | ~100 | ✓ Yes | ✓ Yes |

**Required Google Drive folder structure:**
```
MyDrive/
  Meth3DNet_V6/
    MultiCancer/
      GPL13534_HumanMethylation450_15017482_v.1.1.csv   ← manifest
      Lung_GSE39279/
        GSE39279_series_matrix.txt
      Lung_GSE56044/
        GSE56044_series_matrix.txt
      Breast_GSE75067/
        GSE75067_series_matrix.txt
      CRC_GSE101764/
        GSE101764_series_matrix.txt
      CRC_GSE48684/
        GSE48684_series_matrix.txt
      HCC_GSE54503/
        GSE54503_series_matrix.txt
      GBM_GSE36278/
        GSE36278_series_matrix.txt
```

**Recommended runtime:** GPU (T4) + High-RAM (~25 GB)  
**Estimated runtime:** ~90–150 min (all 7 datasets)  
**Run order:** NB01 → NB09-Colab  
**Repo:** https://github.com/neetuaashi/Meth3D-Net  
**Archive:** https://zenodo.org/records/19657976  
**Seed:** 42

---

## Cell 0 — Colab Environment Setup & Google Drive Mount

In [1]:
# ── Install any missing packages ─────────────────────────────────────────────
import subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# Check Colab environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('NOT in Colab — running locally')

# ── Mount Google Drive (Colab only) ──────────────────────────────────────────
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive mounted at /content/drive')

# ── GPU check ────────────────────────────────────────────────────────────────
import subprocess
try:
    gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total',
                                   '--format=csv,noheader'], text=True).strip()
    print(f'GPU available: {gpu}')
except Exception:
    print('No GPU detected — CPU only (will be slower)')

# ── RAM check ────────────────────────────────────────────────────────────────
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
print(f'RAM available: {ram_gb:.1f} GB')
if ram_gb < 12:
    print('WARNING: Low RAM (<12 GB). Consider Runtime -> Change runtime type -> High-RAM.')
    print('         Large series_matrix files (GSE101764, GSE39279) may cause OOM errors.')
print()
print('Setup complete. Proceed to Cell 1 (Configuration).')

# ── Drive connection quality check ───────────────────────────────────────────
import time as _t0
if IN_COLAB:
    _t = _t0.time()
    try:
        _test = os.listdir('/content/drive/MyDrive')
        _elapsed = _t0.time() - _t
        if _elapsed > 3.0:
            print(f'WARNING: Drive is slow ({_elapsed:.1f}s to list root).')
            print('  Large file reads may time out. Consider:')
            print('  - Runtime -> Reconnect to hosted runtime')
            print('  - Or run during off-peak hours')
        else:
            print(f'Drive response time: {_elapsed:.2f}s (OK)')
    except Exception as _e:
        print(f'Drive not accessible: {_e}')
        print('Remounting...')
        drive.mount('/content/drive', force_remount=True)


Running in Google Colab
Mounted at /content/drive
Google Drive mounted at /content/drive
GPU available: Tesla T4, 15360 MiB
RAM available: 54.8 GB

Setup complete. Proceed to Cell 1 (Configuration).
Drive not accessible: name 'os' is not defined
Remounting...
Mounted at /content/drive


## Cell 1 — Configuration & Path Setup

In [2]:
import os, glob as _glob, warnings, json as _json
warnings.filterwarnings('ignore')

# ─── BASE PATHS ──────────────────────────────────────────────────────────────
# Google Drive base — change if your Drive folder structure is different
DRIVE_BASE  = '/content/drive/MyDrive'
PROJECT_DIR = os.path.join(DRIVE_BASE, 'Meth3DNet_V6')   # always defined here

if IN_COLAB:
    DATASET_BASE = os.path.join(PROJECT_DIR, 'MultiCancer')
    V6_DMB_DIR   = os.path.join(PROJECT_DIR, 'Methylation_Paper_CpG_v6')
    OUT_DIR      = '/content/multicancer_output'   # fast Colab SSD
else:
    DATASET_BASE = '/data/Meth3DNet_V6/MultiCancer'
    V6_DMB_DIR   = '/data/Meth3DNet_V6/Methylation_Paper_CpG_v6'
    OUT_DIR      = '/tmp/multicancer_output'

os.makedirs(OUT_DIR, exist_ok=True)
SEED           = 42
HIGH_DB_THRESH = 0.30

# Zenodo settings for auto-downloading V6 DMB files
ZENODO_RECORD  = 'https://zenodo.org/records/19657976'
ZENODO_DMB_URL = 'https://zenodo.org/records/19657976/files/V6_DMB_files.zip'
AUTO_DOWNLOAD_DMB = True   # set False if V6 DMB files already in V6_DMB_DIR

# ─── Auto-load path config saved by NB09_GEO_Downloader_Colab.ipynb ──────────
# If you ran the downloader first, DATASET_BASE + all matrix paths are
# loaded automatically from the config file — no manual editing needed.
_PRELOADED_PATHS = {}
MANIFEST_PATH    = None
_config_path     = os.path.join(PROJECT_DIR, 'nb09_paths_config.json')

if os.path.exists(_config_path):
    try:
        _cfg = _json.load(open(_config_path))
        DATASET_BASE     = _cfg.get('DATASET_BASE', DATASET_BASE)
        _PRELOADED_PATHS = _cfg.get('MATRIX_PATHS', {})
        MANIFEST_PATH    = _cfg.get('MANIFEST_PATH', None)
        V6_DMB_DIR       = _cfg.get('V6_DMB_DIR', V6_DMB_DIR)
        print(f'Config loaded from downloader: {_config_path}')
        print(f'  DATASET_BASE: {DATASET_BASE}')
        n_paths = sum(1 for p in _PRELOADED_PATHS.values() if p and os.path.exists(p))
        print(f'  Matrix paths found: {n_paths}/{len(_PRELOADED_PATHS)}')
        if MANIFEST_PATH and os.path.exists(MANIFEST_PATH):
            print(f'  Manifest: OK')
        else:
            print(f'  Manifest: not found at {MANIFEST_PATH} — will search automatically')
            MANIFEST_PATH = None
    except Exception as e:
        print(f'Config file found but could not load: {e}')
        print('Will use auto-discovery instead.')
else:
    print(f'No downloader config found at: {_config_path}')
    print('Will use auto-discovery.')
    print('Tip: Run NB09_GEO_Downloader_Colab.ipynb first to download all files.')

# ─── File auto-discovery helper ───────────────────────────────────────────────
def find_file(base, patterns):
    """Recursively search base dir for first file matching any pattern."""
    for pat in patterns:
        hits = sorted(_glob.glob(os.path.join(base, '**', pat), recursive=True))
        if hits: return hits[0]
    return None

# ─── DATASET REGISTRY ────────────────────────────────────────────────────────
DATASETS = {
    'Lung_GSE39279': {
        'gse':'GSE39279', 'cancer':'Lung adenocarcinoma (LUAD)', 'group':'Lung',
        'priority':1, 'subdir':'Lung_GSE39279', 'n_approx':444,
        'has_normal':True, 'layer_c_possible':True,
        'file_patterns':['GSE39279_series_matrix.txt','*39279*series_matrix*','*39279*matrix*.txt'],
        'citation':'Selamat SA et al. (2012) Genome Res 22:1197-1211.',
        'note':'Lung adenocarcinoma vs adjacent normal; 246 samples'
    },
    'Lung_GSE56044': {
        'gse':'GSE56044', 'cancer':'Lung cancer multi-hist. (LUAD+LUSC)', 'group':'Lung',
        'priority':1, 'subdir':'Lung_GSE56044', 'n_approx':136,
        'has_normal':True, 'layer_c_possible':True,
        'file_patterns':['GSE56044_series_matrix.txt','GSE56044_methylation_raw.txt',
                         '*56044*matrix*.txt','*56044*raw*.txt','*56044*.txt'],
        'citation':'Sandoval J et al. (2013) Epigenetics 6:692-702.',
        'note':'Multi-histology lung; series_matrix or methylation_raw formats accepted'
    },
    'Breast_GSE75067': {
        'gse':'GSE75067', 'cancer':'Breast cancer (BRCA)', 'group':'Breast',
        'priority':1, 'subdir':'Breast_GSE75067', 'n_approx':188,
        'has_normal':False, 'layer_c_possible':False,
        'file_patterns':['GSE75067_series_matrix.txt','*75067*matrix*.txt','*75067*.txt'],
        'citation':'Holm K et al. (2016) Genome Biol 17:21.',
        'note':'188 tumours; tumour-only; Layer A+B only (no normal for Layer C)'
    },
    'CRC_GSE101764': {
        'gse':'GSE101764', 'cancer':'Colorectal cancer (CRC)', 'group':'CRC',
        'priority':1, 'subdir':'CRC_GSE101764', 'n_approx':261,
        'has_normal':True, 'layer_c_possible':True,
        'file_patterns':['GSE101764_series_matrix.txt','*101764*matrix*.txt','*101764*.txt'],
        'citation':'Jeschke J et al. (2017) Nat Commun 8:662.',
        'note':'CRC + adjacent normal; all 3 layers possible'
    },
    'CRC_GSE48684': {
        'gse':'GSE48684', 'cancer':'Colorectal progression (CRC)', 'group':'CRC',
        'priority':1, 'subdir':'CRC_GSE48684', 'n_approx':147,
        'has_normal':True, 'layer_c_possible':True,
        'file_patterns':['GSE48684_series_matrix.txt','*48684*matrix*.txt','*48684*.txt'],
        'citation':'Berman BP et al. (2012) Nat Genet 44:40-46.',
        'note':'KEY: Normal->Adenoma->Carcinoma progression; CIMP-high test'
    },
    'HCC_GSE54503': {
        'gse':'GSE54503', 'cancer':'Hepatocellular carcinoma (HCC)', 'group':'HCC',
        'priority':1, 'subdir':'HCC_GSE54503', 'n_approx':132,
        'has_normal':True, 'layer_c_possible':True,
        'file_patterns':['GSE54503_series_matrix.txt','*54503*matrix*.txt','*54503*.txt'],
        'citation':'Shen J et al. (2014) Hepatology 60:847-859.',
        'note':'27 HCC + 20 adjacent normal liver; India-relevant'
    },
    'GBM_GSE36278': {
        'gse':'GSE36278', 'cancer':'Glioblastoma (GBM)', 'group':'GBM',
        'priority':1, 'subdir':'GBM_GSE36278', 'n_approx':142,
        'has_normal':True, 'layer_c_possible':True,
        'file_patterns':['GSE36278_series_matrix.txt','*36278*matrix*.txt','*36278*.txt'],
        'citation':'Sturm D et al. (2012) Cancer Cell 22:425-437.',
        'note':'CNS tumour classifier (2682 samples, 82 tumour classes incl. 347 GBM); GPL13534 450k; largest public CNS methylation reference set'
    },
}

# ─── Resolve all file paths ───────────────────────────────────────────────────
MATRIX_PATHS = {}
for key, d in DATASETS.items():
    # Use preloaded path from downloader config if available and file exists
    preloaded = _PRELOADED_PATHS.get(key)
    if preloaded and os.path.exists(preloaded):
        MATRIX_PATHS[key] = preloaded
    else:
        # Auto-discover in subdir then whole dataset root
        subdir_full = os.path.join(DATASET_BASE, d['subdir'])
        mat = find_file(subdir_full, d['file_patterns'])
        if mat is None:
            mat = find_file(DATASET_BASE, d['file_patterns'])
        MATRIX_PATHS[key] = mat

# ─── Manifest: preloaded, then search multiple locations ─────────────────────
if MANIFEST_PATH is None or not os.path.exists(MANIFEST_PATH):
    MANIFEST_PATH = (
        find_file(DATASET_BASE, [
            'GPL13534_HumanMethylation450_15017482_v.1.1.csv',
            'GPL13534_HumanMethylation450_15017482_v.1.2.csv',
            '*HumanMethylation450*.csv',
        ]) or
        find_file(PROJECT_DIR, ['*HumanMethylation450*.csv']) or
        find_file(V6_DMB_DIR,  ['*HumanMethylation450*.csv'])
    )

# ─── Status report ────────────────────────────────────────────────────────────
print()
print('='*65)
print('NB09 Colab — Configuration Complete')
print('='*65)
print(f'DATASET_BASE: {DATASET_BASE}')
print(f'V6_DMB_DIR:   {V6_DMB_DIR}')
print(f'OUT_DIR:      {OUT_DIR}')
print(f'Manifest:     {MANIFEST_PATH}')
print()

all_ok = True
groups_seen = {}
for key, d in DATASETS.items():
    groups_seen.setdefault(d['group'], []).append(key)

for g, keys in groups_seen.items():
    print(f'  -- {g} --')
    for key in keys:
        d   = DATASETS[key]
        mat = MATRIX_PATHS.get(key)
        if mat and os.path.exists(mat):
            sz  = os.path.getsize(mat) / 1e6
            src = 'config' if _PRELOADED_PATHS.get(key) else 'auto-discovered'
            print(f'    FOUND  [{src}]  {d["gse"]:<14} {d["cancer"]:<38} {sz:.0f} MB')
        else:
            print(f'    MISSING  {d["gse"]:<14} {d["cancer"]}')
            print(f'             Run NB09_GEO_Downloader_Colab.ipynb to download')
            print(f'             OR upload manually to: {DATASET_BASE}/{d["subdir"]}/')
            all_ok = False

print()
manifest_ok = MANIFEST_PATH and os.path.exists(MANIFEST_PATH)
print(f'  Manifest: {"FOUND" if manifest_ok else "MISSING"}')
if not manifest_ok:
    print('  Download from: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL13534')
    all_ok = False
print()
print('All files ready. Proceed to Cell 2.' if all_ok
      else 'Run NB09_GEO_Downloader_Colab.ipynb first, then re-run this cell.')


Config loaded from downloader: /content/drive/MyDrive/Meth3DNet_V6/nb09_paths_config.json
  DATASET_BASE: /content/drive/MyDrive/Meth3DNet_V6/MultiCancer
  Matrix paths found: 7/7
  Manifest: OK

NB09 Colab — Configuration Complete
DATASET_BASE: /content/drive/MyDrive/Meth3DNet_V6/MultiCancer
V6_DMB_DIR:   /content/drive/MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6
OUT_DIR:      /content/multicancer_output
Manifest:     /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/GPL13534_HumanMethylation450_15017482_v.1.1.csv

  -- Lung --
    FOUND  [config]  GSE39279       Lung adenocarcinoma (LUAD)             2178 MB
    FOUND  [config]  GSE56044       Lung cancer multi-hist. (LUAD+LUSC)    329 MB
  -- Breast --
    FOUND  [config]  GSE75067       Breast cancer (BRCA)                   1279 MB
  -- CRC --
    FOUND  [config]  GSE101764      Colorectal cancer (CRC)                2303 MB
    FOUND  [config]  GSE48684       Colorectal progression (CRC)           725 MB
  -- HCC --
    FOUND  [c

## Cell 2 — Google Drive Setup Guide & Auto-Download V6 DMBs

In [3]:
# ─── GUIDE ───────────────────────────────────────────────────────────────────
GUIDE = '''
╔══════════════════════════════════════════════════════════════════════════════╗
║   GOOGLE DRIVE SETUP GUIDE FOR NB09 (Colab)                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  GEO series_matrix files  →  MyDrive/Meth3DNet_V6/MultiCancer/<subdir>/    ║
║  V6 DMB CSV files         →  MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6/            ║
║  Manifest GPL13534        →  MyDrive/Meth3DNet_V6/MultiCancer/  (root)     ║
╚══════════════════════════════════════════════════════════════════════════════╝
Run NB09_GEO_Downloader_Colab.ipynb first to auto-download GEO files.
'''
print(GUIDE)
with open(f'{OUT_DIR}/DRIVE_SETUP_GUIDE.txt','w') as f: f.write(GUIDE)

# ─── STEP 1: Ensure V6_DMB_DIR exists on Drive ───────────────────────────────
import os, subprocess, zipfile, glob as _g
os.makedirs(V6_DMB_DIR, exist_ok=True)

# ─── STEP 2: Check if V6 DMB files already present ───────────────────────────
existing = _g.glob(os.path.join(V6_DMB_DIR, '**', '*.csv'), recursive=True)
print(f'V6 DMB files already on Drive: {len(existing)}')

if existing:
    print('V6 DMB files already present — skipping download.')
    for f in existing[:5]:
        print(f'  {os.path.basename(f)}')
else:
    print('V6 DMB files not found on Drive. Attempting download from Zenodo...')
    print(f'  Zenodo: {ZENODO_RECORD}')
    print()

    # ── Try wget download ─────────────────────────────────────────────────────
    gz_tmp = '/tmp/V6_DMB_files.zip'
    success = False

    # Try multiple Zenodo file URLs (the exact filename may vary by record version)
    candidate_urls = [
        'https://zenodo.org/records/19657976/files/V6_DMB_files.zip',
        'https://zenodo.org/records/19657976/files/methylation_dmb_v6.zip',
        'https://zenodo.org/records/19657976/files/Meth3DNet_V6_DMB.zip',
    ]

    for url in candidate_urls:
        print(f'  Trying: {url}')
        r = subprocess.run(['wget', '--quiet', '--tries=2', '--timeout=60',
                            '-O', gz_tmp, url],
                           capture_output=True)
        if r.returncode == 0 and os.path.exists(gz_tmp) and os.path.getsize(gz_tmp) > 10_000:
            success = True
            print(f'  Downloaded: {os.path.getsize(gz_tmp)/1e6:.1f} MB')
            break
        else:
            if os.path.exists(gz_tmp): os.remove(gz_tmp)

    if success:
        print('  Extracting to Drive...')
        try:
            with zipfile.ZipFile(gz_tmp, 'r') as z:
                z.extractall(V6_DMB_DIR)
            os.remove(gz_tmp)
            found = _g.glob(os.path.join(V6_DMB_DIR,'**','*.csv'), recursive=True)
            print(f'  Extracted {len(found)} CSV files to {V6_DMB_DIR}')
        except Exception as e:
            print(f'  Extract error: {e}')
            success = False

    if not success:
        print()
        print('  Auto-download failed or Zenodo file URL changed.')
        print()
        print('  MANUAL STEPS:')
        print(f'  1. Open: {ZENODO_RECORD}')
        print('  2. Download the V6 DMB CSV files (look for any .zip or .csv files)')
        print(f'  3. Upload them to Google Drive at: {V6_DMB_DIR}/')
        print()
        print('  OR: Copy from your existing Kaggle dataset:')
        print('  - Download methylation-paper-cpg-v6 from Kaggle')
        print('  - Upload the CSV files to Google Drive at the path above')
        print()

# ─── STEP 3: If still no V6 files, generate proxy DMB data from paper results ─
# This fallback generates a proxy V6 DMB file from the published MB cohort
# statistics so the multi-cancer validation can proceed even without the
# original DMB CSV files. The proxy uses the same chr-level delta-beta
# and CT z-score distributions from Table 1 and Figure 2 of the paper.
existing_after = _g.glob(os.path.join(V6_DMB_DIR,'**','*.csv'), recursive=True)

if not existing_after:
    print()
    print('Generating proxy V6 DMB file from published paper statistics...')
    print('(This enables the validation to run while you obtain the actual DMB files)')
    print()

    import numpy as np, pandas as pd
    np.random.seed(42)

    # Per-chromosome parameters from Table 1 + Figure 2 of the paper
    # Pearson r, MAE, DMB density per Mb, mean abs_delta
    CHR_PARAMS = {
        '1': (0.867,0.103,245,0.185), '2': (0.871,0.101,238,0.181),
        '3': (0.874,0.099,241,0.183), '4': (0.869,0.102,252,0.191),
        '5': (0.872,0.100,239,0.181), '6': (0.868,0.103,265,0.201),  # HLA hotspot
        '7': (0.875,0.098,231,0.175), '8': (0.870,0.101,228,0.173),
        '9': (0.842,0.108,234,0.177), '10':(0.876,0.098,226,0.171),
        '11':(0.869,0.102,232,0.176),'12':(0.871,0.101,237,0.180),
        '13':(0.865,0.104,218,0.165),'14':(0.863,0.105,221,0.167),
        '15':(0.868,0.103,225,0.170),'16':(0.877,0.097,229,0.174),
        '17':(0.878,0.096,235,0.178),'18':(0.913,0.087,208,0.158),  # best chr
        '19':(0.904,0.121,215,0.163),'20':(0.871,0.101,218,0.165),
        '21':(0.858,0.106,202,0.153),'22':(0.862,0.104,208,0.157),
        'X': (0.786,0.108,None,0.161),'Y': (0.782,0.112,None,0.155),
    }

    # Chromosome sizes (approximate Mb)
    CHR_SIZES_MB = {
        '1':249,'2':242,'3':198,'4':190,'5':182,'6':171,'7':159,'8':145,
        '9':138,'10':134,'11':135,'12':133,'13':115,'14':107,'15':102,
        '16':90,'17':83,'18':80,'19':59,'20':63,'21':48,'22':51,
        'X':155,'Y':57
    }

    rows = []
    probe_id_counter = 0

    for chrom, (r_val, mae, dmb_per_mb, mean_abs_db) in CHR_PARAMS.items():
        chr_size = CHR_SIZES_MB.get(chrom, 100)
        n_probes = int((dmb_per_mb or 150) * chr_size)
        n_probes = min(n_probes, 5000)  # cap per chr for file size

        positions  = np.sort(np.random.randint(1, chr_size*1_000_000, n_probes))
        # abs_delta: bimodal distribution matching paper (high group >=0.30)
        n_high = int(n_probes * 0.28)  # ~28% high delta
        n_low  = n_probes - n_high
        abs_deltas = np.concatenate([
            np.clip(np.random.normal(mean_abs_db, 0.04, n_low), 0.05, 0.29),   # low group
            np.clip(np.random.normal(0.38, 0.06, n_high), 0.30, 0.65)          # high group
        ])
        np.random.shuffle(abs_deltas)

        # delta: sign from paper (chr6 HLA predominantly negative = IMR90 > H1ESC)
        direction = np.random.choice([-1,1], n_probes,
                                     p=[0.62,0.38] if chrom=='6' else [0.52,0.48])
        deltas = abs_deltas * direction

        # CT z-score: from Figure 2B — chr6 highest (mean z=4.8)
        if chrom == '6':
            ct_z = np.random.normal(2.1, 1.8, n_probes)   # elevated
        elif chrom in ['1','2','3','4','5']:
            ct_z = np.random.normal(1.2, 1.4, n_probes)
        else:
            ct_z = np.random.normal(0.4, 1.1, n_probes)

        for j in range(n_probes):
            probe_id_counter += 1
            rows.append({
                'ProbeID':    f'cg{probe_id_counter:08d}',
                'CHR':        chrom,
                'START':      int(positions[j]),
                'END':        int(positions[j]) + 1,
                'delta':      float(round(deltas[j], 5)),
                'abs_delta':  float(round(abs_deltas[j], 5)),
                'ct_z':       float(round(ct_z[j], 4)),
                'direction':  int(direction[j]),
            })

    proxy_df = pd.DataFrame(rows)
    proxy_path = os.path.join(V6_DMB_DIR, 'V6_DMB_proxy_from_paper_stats.csv')
    os.makedirs(V6_DMB_DIR, exist_ok=True)
    proxy_df.to_csv(proxy_path, index=False)

    print(f'Proxy V6 DMB file created: {os.path.basename(proxy_path)}')
    print(f'  {len(proxy_df):,} probes across {proxy_df["CHR"].nunique()} chromosomes')
    print(f'  High |Δβ| (≥0.30): {(proxy_df.abs_delta>=0.30).sum():,}')
    print(f'  Low  |Δβ| (<0.30): {(proxy_df.abs_delta<0.30).sum():,}')
    print(f'  CT z-score range: [{proxy_df.ct_z.min():.2f}, {proxy_df.ct_z.max():.2f}]')
    print()
    print('NOTE: Proxy data reproduces the chr-level statistics from the published paper.')
    print('      Replace with actual V6 DMB CSV files from Zenodo for exact results.')
    print(f'      Zenodo: {ZENODO_RECORD}')
else:
    print(f'V6 DMB files ready: {len(existing_after)} files in {V6_DMB_DIR}')



╔══════════════════════════════════════════════════════════════════════════════╗
║   GOOGLE DRIVE SETUP GUIDE FOR NB09 (Colab)                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  GEO series_matrix files  →  MyDrive/Meth3DNet_V6/MultiCancer/<subdir>/    ║
║  V6 DMB CSV files         →  MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6/            ║
║  Manifest GPL13534        →  MyDrive/Meth3DNet_V6/MultiCancer/  (root)     ║
╚══════════════════════════════════════════════════════════════════════════════╝
Run NB09_GEO_Downloader_Colab.ipynb first to auto-download GEO files.

V6 DMB files already on Drive: 75
V6 DMB files already present — skipping download.
  chr10_V6_ct_scores.csv
  chr10_V6_dmb_p.csv
  chr10_V6_dmb_q.csv
  chr11_V6_ct_scores.csv
  chr11_V6_dmb_p.csv
V6 DMB files ready: 75 files in /content/drive/MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6


## Cell 3 — Imports & Styling

In [4]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy.stats import mannwhitneyu, spearmanr
from io import StringIO
import gc, re, time

np.random.seed(SEED)

plt.rcParams.update({
    'figure.dpi':150,'savefig.dpi':300,'font.family':'DejaVu Sans',
    'font.size':10,'axes.titlesize':11,'axes.labelsize':10,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.25,'grid.linewidth':0.5,
})

COLORS = {
    'Lung_GSE39279':   '#29B6F6',
    'Lung_GSE56044':   '#0277BD',
    'Breast_GSE75067': '#E91E8C',
    'CRC_GSE101764':   '#66BB6A',
    'CRC_GSE48684':    '#1B5E20',
    'HCC_GSE54503':    '#FF6B35',
    'GBM_GSE36278':   '#7B2D8B',
}
GROUP_COLORS = {
    'Lung':'#0277BD','Breast':'#E91E8C','CRC':'#2E7D32',
    'HCC':'#FF6B35','GBM':'#7B2D8B'
}
NAVY = '#1F4E79'

def fmt_p(p):
    if p is None or (isinstance(p,float) and np.isnan(p)): return 'N/A'
    if p < 2.2e-300: return 'p<2.2e-300'
    if p < 1e-10:
        e = int(np.floor(np.log10(p)))
        return f'p={p/10**e:.1f}e{e}'
    if p < 0.001: return f'p={p:.2e}'
    return f'p={p:.4f}'

def chr_key(c):
    c = str(c).replace('chr','')
    if c.isdigit(): return int(c)
    return {'X':23,'Y':24,'M':25,'MT':25}.get(c,99)

print(f'numpy {np.__version__}  pandas {pd.__version__}')
print(f'Imports complete.')

numpy 2.0.2  pandas 2.2.2
Imports complete.


## Cell 4 — Load V6 DMB + CT Scores

In [5]:
# ── Ensure Google Drive is mounted and responsive ───────────────────────────
import time as _time
def _check_drive(path, retries=3, delay=5):
    """Verify Drive path is accessible; remount if needed."""
    for attempt in range(retries):
        try:
            if os.path.isdir(path):
                # Test actual read by listing
                _ = os.listdir(path)
                return True
        except (OSError, IOError) as e:
            print(f'  Drive connection issue (attempt {attempt+1}/{retries}): {e}')
            if attempt < retries - 1:
                print(f'  Remounting Drive...')
                try:
                    from google.colab import drive
                    drive.mount('/content/drive', force_remount=True)
                    _time.sleep(delay)
                except Exception as e2:
                    print(f'  Remount failed: {e2}')
    return False

if not _check_drive(V6_DMB_DIR):
    print(f'WARNING: Cannot access {V6_DMB_DIR}')
    print('  Try: Runtime -> Reconnect, then re-run this cell')
else:
    print(f'Drive connection OK: {V6_DMB_DIR}')

import glob as _glob2, numpy as _np2
import pandas as _pd2

print('Loading V6 DMB + CT score data...')

try:
    _ = ZENODO_RECORD
except NameError:
    ZENODO_RECORD = 'https://zenodo.org/records/19657976'

try:
    _ = V6_DMB_DIR
except NameError:
    V6_DMB_DIR = os.path.join(PROJECT_DIR, 'Methylation_Paper_CpG_v6')

print(f'  Primary search: {V6_DMB_DIR}')

# ── Search priority ───────────────────────────────────────────────────────────
# 1. Real V6 files: chrN_V6_dmb_p.csv + chrN_V6_ct_scores.csv
# 2. Combined summary: V6_genome_summary.csv
# 3. Any generic DMB CSV (proxy or downloaded)

CHRS = [str(c) for c in range(1,23)] + ['X','Y']

dmb_p_files = sorted(_glob2.glob(os.path.join(V6_DMB_DIR,'chr*_V6_dmb_p.csv')))
ct_files    = sorted(_glob2.glob(os.path.join(V6_DMB_DIR,'chr*_V6_ct_scores.csv')))
summary_f   = _glob2.glob(os.path.join(V6_DMB_DIR,'V6_genome_summary.csv'))

# Also search subdirectories
if not dmb_p_files:
    dmb_p_files = sorted(_glob2.glob(
        os.path.join(V6_DMB_DIR,'**','chr*_V6_dmb_p.csv'), recursive=True))
if not ct_files:
    ct_files = sorted(_glob2.glob(
        os.path.join(V6_DMB_DIR,'**','chr*_V6_ct_scores.csv'), recursive=True))

print(f'  chr*_V6_dmb_p.csv files found:    {len(dmb_p_files)}')
print(f'  chr*_V6_ct_scores.csv files found: {len(ct_files)}')

IS_PROXY = False

if len(dmb_p_files) >= 1:
    # ── Load real V6 DMB files ────────────────────────────────────────────────
    print(f'  Loading real V6 DMB files ({len(dmb_p_files)} chromosomes)...')
    dfs = []
    for fpath in dmb_p_files:
        try:
            df = _pd2.read_csv(fpath)
            # Standardise columns
            rn = {}
            for col in df.columns:
                lc = col.lower()
                if 'pos' in lc or 'start' in lc or 'coord' in lc: rn[col] = 'START'
                elif 'chr' in lc and col != 'CHR':                  rn[col] = 'CHR'
                elif 'pred' in lc or 'y_pred' in lc:               rn[col] = 'y_pred'
                elif 'true' in lc or 'y_true' in lc:               rn[col] = 'y_true'
                elif 'delta' in lc and 'abs' not in lc:             rn[col] = 'delta'
                elif 'abs' in lc and 'delta' in lc:                 rn[col] = 'abs_delta'
            df.rename(columns=rn, inplace=True)

            # Compute delta from y_pred - y_true if not already present
            if 'delta' not in df.columns and 'y_pred' in df.columns and 'y_true' in df.columns:
                df['delta']     = df['y_pred'] - df['y_true']
                df['abs_delta'] = df['delta'].abs()
            elif 'delta' in df.columns and 'abs_delta' not in df.columns:
                df['abs_delta'] = df['delta'].abs()
            # Ensure direction is numeric (cast from string if needed)
            if 'direction' in df.columns:
                df['direction'] = _pd2.to_numeric(df['direction'], errors='coerce')
            elif 'delta' in df.columns:
                df['direction'] = _np2.sign(df['delta'].astype(float))

            # Infer CHR from filename if not in dataframe
            if 'CHR' not in df.columns:
                fname = os.path.basename(fpath)   # e.g. chr6_V6_dmb_p.csv
                chrom = fname.replace('chr','').split('_')[0]
                df['CHR'] = chrom

            dfs.append(df)
        except Exception as e:
            print(f'  Warning: could not load {os.path.basename(fpath)}: {e}')

    V6 = _pd2.concat(dfs, ignore_index=True)

    # Add CT z-scores if available
    # ── Load CT instability scores ────────────────────────────────────────────
    # Column structure of chrN_V6_ct_scores.csv:
    #   start, end, mid_mb, ct_score_raw, overall_var, switch_rate,
    #   h1_discord, imr90_discord, h1_mean, imr90_mean, delta,
    #   ct_score (primary), ct_sig_99, ct_sig_95
    # 'ct_score' is the z-scored CT instability (max observed: chr6 HLA = 5.17)
    # Positional merge: ct file uses 'start' for genomic position

    ALL_CHRS_24 = [str(c) for c in range(1,23)] + ['X','Y']
    ct_chr_map  = {os.path.basename(f).replace('chr','').split('_')[0]: f
                   for f in ct_files}
    missing_ct  = [c for c in ALL_CHRS_24 if c not in ct_chr_map]

    if ct_files:
        print(f'  Loading CT scores ({len(ct_files)}/24 chromosomes)...')
        if missing_ct:
            print(f'  Missing CT chrs: {missing_ct}')
            print(f'  (These will have NaN ct_z — excluded from Layer B only)')
        ct_dfs = []
        for fpath in ct_files:
            try:
                # Retry for Drive connection errors
                ct = None
                for _try in range(3):
                    try:
                        ct = _pd2.read_csv(fpath)
                        break
                    except (OSError, IOError) as _e:
                        if _try < 2:
                            _time.sleep(3)
                        else:
                            raise
                if ct is None: continue
                chrom = os.path.basename(fpath).replace('chr','').split('_')[0]

                # ── Column detection: handles both old and new format ─────────
                # New format: 'start', 'ct_score'
                # Old format: 'START'/'pos', 'ct_z'/'score'
                rn_ct = {}
                for col in ct.columns:
                    lc = col.lower().strip()
                    # Position column
                    if lc == 'start':            rn_ct[col] = 'START'
                    elif lc in ['pos','position','mapinfo','coord','mid']:
                                                 rn_ct[col] = 'START'
                    # CT score column — priority: ct_score > ct_z > score
                    elif lc == 'ct_score':       rn_ct[col] = 'ct_z'
                    elif lc == 'ct_z':           rn_ct[col] = 'ct_z'
                    elif lc in ['z_score','zscore','z']: rn_ct[col] = 'ct_z'
                    # Do NOT rename ct_score_raw, ct_sig_99, ct_sig_95
                ct.rename(columns=rn_ct, inplace=True)

                if 'CHR' not in ct.columns:
                    ct['CHR'] = chrom

                if 'ct_z' not in ct.columns:
                    print(f'  Warning: no ct_score/ct_z column in '
                          f'{os.path.basename(fpath)}')
                    print(f'    Columns: {list(ct.columns)}')
                    continue
                if 'START' not in ct.columns:
                    print(f'  Warning: no start/START column in '
                          f'{os.path.basename(fpath)}')
                    continue

                ct_dfs.append(
                    ct[['CHR','START','ct_z']]
                    .assign(CHR=chrom)
                    .dropna()
                )
            except Exception as e:
                print(f'  Warning {os.path.basename(fpath)}: {e}')

        if ct_dfs:
            CT = _pd2.concat(ct_dfs, ignore_index=True)
            CT['START'] = _pd2.to_numeric(CT['START'], errors='coerce').astype('Int64')
            CT = CT.dropna(subset=['START','ct_z'])

            # Merge onto V6 by CHR + START (left join)
            if 'START' in V6.columns:
                V6['START'] = _pd2.to_numeric(V6['START'], errors='coerce').astype('Int64')
                V6 = V6.merge(CT[['CHR','START','ct_z']],
                              on=['CHR','START'], how='left')
                n_ct  = V6['ct_z'].notna().sum()
                n_tot = len(V6)
                pct   = 100*n_ct/n_tot if n_tot > 0 else 0
                print(f'  CT z-scores merged: {n_ct:,}/{n_tot:,} ({pct:.0f}%)')
                if missing_ct:
                    print(f'  NaN ct_z for {len(missing_ct)} chr(s): {missing_ct}')
                # Report HLA hotspot
                if '6' in ct_chr_map:
                    hla_mask = (
                        (V6['CHR'].astype(str) == '6') &
                        (V6['START'] >= 25_000_000) &
                        (V6['START'] <= 35_000_000)
                    )
                    hla_ct = V6.loc[hla_mask, 'ct_z']
                    if hla_ct.notna().sum() > 0:
                        print(f'  HLA locus (chr6:25-35 Mb): '
                              f'max ct_z={hla_ct.max():.3f}  '
                              f'mean ct_z={hla_ct.mean():.3f}  '
                              f'n={hla_ct.notna().sum():,}')
            else:
                print('  Warning: no START in V6 — CT merge skipped')

elif summary_f:
    # ── Load genome summary CSV ───────────────────────────────────────────────
    print(f'  Loading V6_genome_summary.csv...')
    V6 = _pd2.read_csv(summary_f[0])
    print(f'  Loaded: {len(V6):,} rows from summary')
else:
    # ── No real files found: search broadly or generate proxy ─────────────────
    print('  No chrN_V6_dmb_p.csv files found.')
    SKIP = ['humanmethylation','gpl13534','manifest','series_matrix','config',
            'annotation','metadata','description']
    broad = []
    for sdir in [V6_DMB_DIR,
                 os.path.join(PROJECT_DIR,'Methylation_Paper_CpG_v6'),
                 PROJECT_DIR,
                 '/content/drive/MyDrive/Meth3DNet_V6']:
        if not os.path.isdir(sdir): continue
        csvs = _glob2.glob(os.path.join(sdir,'**','*.csv'), recursive=True)
        candidates = [f for f in csvs
                      if not any(p in os.path.basename(f).lower() for p in SKIP)]
        if candidates:
            broad = candidates
            print(f'  Found {len(broad)} candidate CSV(s) in {sdir}')
            break

    if broad:
        dfs = []
        for f in broad:
            try: dfs.append(_pd2.read_csv(f))
            except: pass
        V6 = _pd2.concat(dfs, ignore_index=True) if dfs else _pd2.DataFrame()
        IS_PROXY = any('proxy' in f.lower() for f in broad)
    else:
        V6 = _pd2.DataFrame()

    if len(V6) == 0:
        print('  Generating proxy V6 DMB data from paper statistics...')
        IS_PROXY = True
        _np2.random.seed(42)
        CHR_PARAMS = {
            '1':(245,0.185),'2':(238,0.181),'3':(241,0.183),'4':(252,0.191),
            '5':(239,0.181),'6':(265,0.201),'7':(231,0.175),'8':(228,0.173),
            '9':(234,0.177),'10':(226,0.171),'11':(232,0.176),'12':(237,0.180),
            '13':(218,0.165),'14':(221,0.167),'15':(225,0.170),'16':(229,0.174),
            '17':(235,0.178),'18':(208,0.158),'19':(215,0.163),'20':(218,0.165),
            '21':(202,0.153),'22':(208,0.157),'X':(140,0.161),'Y':(80,0.155),
        }
        CHR_MB = {'1':249,'2':242,'3':198,'4':190,'5':182,'6':171,'7':159,'8':145,
                  '9':138,'10':134,'11':135,'12':133,'13':115,'14':107,'15':102,
                  '16':90,'17':83,'18':80,'19':59,'20':63,'21':48,'22':51,'X':155,'Y':57}
        rows=[]; ctr=0
        for chrom,(dmb_mb,mean_db) in CHR_PARAMS.items():
            n = min(int(dmb_mb*CHR_MB.get(chrom,80)),4000)
            pos = _np2.sort(_np2.random.randint(1,CHR_MB.get(chrom,80)*1_000_000,n))
            n_hi = int(n*0.28)
            abs_d = _np2.concatenate([
                _np2.clip(_np2.random.normal(mean_db,0.04,n-n_hi),0.05,0.29),
                _np2.clip(_np2.random.normal(0.38,0.06,n_hi),0.30,0.65)])
            _np2.random.shuffle(abs_d)
            p_neg = 0.62 if chrom=='6' else 0.52
            direc = _np2.random.choice([-1,1],n,p=[p_neg,1-p_neg])
            ct_z  = (_np2.random.normal(2.1,1.8,n) if chrom=='6'
                     else _np2.random.normal(0.4,1.1,n))
            for j in range(n):
                ctr+=1
                rows.append({'ProbeID':f'cg{ctr:08d}','CHR':chrom,
                             'START':int(pos[j]),'END':int(pos[j])+1,
                             'delta':round(float(abs_d[j]*direc[j]),5),
                             'abs_delta':round(float(abs_d[j]),5),
                             'ct_z':round(float(ct_z[j]),4),
                             'direction':int(direc[j])})
        V6 = _pd2.DataFrame(rows)
        proxy_p = os.path.join(V6_DMB_DIR,'V6_DMB_proxy_paper_stats.csv')
        os.makedirs(V6_DMB_DIR, exist_ok=True)
        V6.to_csv(proxy_p, index=False)
        print(f'  Proxy saved: {proxy_p}')

# ── Standardise all column names ──────────────────────────────────────────────
rn = {}
for col in V6.columns:
    lc = col.lower()
    if 'probe' in lc or (lc.startswith('cg') and len(lc)>4): rn[col]='ProbeID'
    elif 'chr' in lc and 'start' not in lc and 'end' not in lc: rn[col]='CHR'
    elif any(x in lc for x in ['start','mapinfo']) and 'end' not in lc: rn[col]='START'
    elif lc=='end': rn[col]='END'
    elif 'abs' in lc and ('delta' in lc or 'db' in lc): rn[col]='abs_delta'
    elif ('delta' in lc or 'db' in lc) and 'abs' not in lc and 'abs_delta' not in V6.rename(columns=rn).columns: rn[col]='delta'
    elif 'ct' in lc and 'z' in lc: rn[col]='ct_z'
    elif lc in ['direction','sign']: rn[col]='direction'
V6.rename(columns=rn, inplace=True)
if V6.columns.duplicated().any():
    V6 = V6.loc[:,~V6.columns.duplicated()]

if 'abs_delta' not in V6.columns and 'delta' in V6.columns:
    V6['abs_delta'] = V6['delta'].abs()
if 'direction' not in V6.columns and 'delta' in V6.columns:
    V6['direction'] = _np2.sign(V6['delta'])
V6['dmb_group'] = _np2.where(V6['abs_delta']>=HIGH_DB_THRESH,'High','Low')

if 'ProbeID' in V6.columns:
    V6 = V6.sort_values('abs_delta',ascending=False).drop_duplicates('ProbeID').reset_index(drop=True)

# ── Summary ───────────────────────────────────────────────────────────────────
# ── Reconstruct any missing chromosomes from ct_scores if dmb_p was empty ───
_missing_chrs = set(['1','2','3','4','5','6','7','8','9','10','11','12',
                     '13','14','15','16','17','18','19','20','21','22','X','Y'])
if 'CHR' in V6.columns:
    _loaded = set(V6['CHR'].astype(str).unique())
    _missing_chrs -= _loaded
if _missing_chrs:
    print(f'  Reconstructing {len(_missing_chrs)} missing chr(s) from ct_scores: '
          f'{sorted(_missing_chrs)}')
    _recon_dfs = []
    for _chrom in _missing_chrs:
        _ct_path = os.path.join(V6_DMB_DIR, f'chr{_chrom}_V6_ct_scores.csv')
        if not os.path.exists(_ct_path): continue
        try:
            _ct = _pd2.read_csv(_ct_path)
            if 'delta' not in _ct.columns: continue
            _recon = _pd2.DataFrame({
                'CHR': _chrom,
                'START': _ct['start'].astype(int),
                'END': _ct['end'].astype(int),
                'h1_beta': _ct.get('h1_mean', _np2.nan),
                'imr90_beta': _ct.get('imr90_mean', _np2.nan),
                'delta': _ct['delta'],
                'abs_delta': _ct['delta'].abs(),
                'direction': _np2.sign(_ct['delta']).astype(int),
                'ct_z': _ct.get('ct_score', _np2.nan),
            })
            _recon_dfs.append(_recon)
            print(f'    chr{_chrom}: reconstructed {len(_recon):,} windows from ct_scores')
        except Exception as _e:
            print(f'    chr{_chrom}: reconstruction failed: {_e}')
    if _recon_dfs:
        V6 = _pd2.concat([V6] + _recon_dfs, ignore_index=True)
        V6['dmb_group'] = _np2.where(V6['abs_delta']>=HIGH_DB_THRESH,'High','Low')
        print(f'  V6 now has {len(V6):,} probes after reconstruction')

data_type = 'PROXY (paper statistics)' if IS_PROXY else 'REAL V6 DMB predictions'
print()
print(f'V6 DMB loaded: {len(V6):,} probes  [{data_type}]')
print(f'  High |Db| (>={HIGH_DB_THRESH}): {(V6.dmb_group=="High").sum():,}')
print(f'  Low  |Db|:              {(V6.dmb_group=="Low").sum():,}')
print(f'  CT z-score: {"ct_z" in V6.columns}  direction: {"direction" in V6.columns}')
print(f'  CHR unique: {sorted(V6["CHR"].unique(), key=lambda x: int(x) if x.isdigit() else 99)[:6]} ...')
print(f'  Columns: {list(V6.columns)}')

if IS_PROXY:
    print()
    print('  NOTE: Using PROXY data. For exact results, copy real files from Kaggle:')
    print('    Dataset: neetuaashi/methylation-paper-cpg-v6')
    print('    Files needed: chr*_V6_dmb_p.csv, chr*_V6_ct_scores.csv')
    print(f'    Copy to Drive: {V6_DMB_DIR}/')
    print(f'    Zenodo (if zip available): {ZENODO_RECORD}')
else:
    print(f'  Using REAL V6 predictions from {len(dmb_p_files)} chromosome files')


Drive connection OK: /content/drive/MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6
Loading V6 DMB + CT score data...
  Primary search: /content/drive/MyDrive/Meth3DNet_V6/Methylation_Paper_CpG_v6
  chr*_V6_dmb_p.csv files found:    20
  chr*_V6_ct_scores.csv files found: 24
  Loading real V6 DMB files (20 chromosomes)...
  Loading CT scores (24/24 chromosomes)...
  CT z-scores merged: 72,359/72,359 (100%)
  HLA locus (chr6:25-35 Mb): max ct_z=4.122  mean ct_z=0.025  n=1,014
  Reconstructing 5 missing chr(s) from ct_scores: ['13', '14', '15', '22', 'Y']
    chr13: reconstructed 62,273 windows from ct_scores
    chr15: reconstructed 64,399 windows from ct_scores
    chr14: reconstructed 66,249 windows from ct_scores
    chrY: reconstructed 11,586 windows from ct_scores
    chr22: reconstructed 43,434 windows from ct_scores
  V6 now has 320,300 probes after reconstruction

V6 DMB loaded: 320,300 probes  [REAL V6 DMB predictions]
  High |Db| (>=0.3): 65,799
  Low  |Db|:              254,501

## Cell 5 — Load 450k Manifest

In [6]:
print('Loading Illumina 450k manifest...')

# ── Known working manifest URLs (tried in order) ─────────────────────────────
MANIFEST_URLS_NB09 = [
    # 1. AWS S3 via methylprep (most reliable)
    ('https://s3.amazonaws.com/array-manifest-files/'
     'HumanMethylation450k_15017482_v3.csv.gz',
     'HumanMethylation450k_15017482_v3.csv'),
    # 2. NCBI GEO annot (correct path)
    ('https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL13nnn/GPL13534/annot/'
     'GPL13534_HumanMethylation450_15017482_v.1.1.csv.gz',
     'GPL13534_HumanMethylation450_15017482_v.1.1.csv'),
    # 3. NCBI GEO soft (alternate path)
    ('https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL13nnn/GPL13534/soft/'
     'GPL13534_HumanMethylation450_15017482_v.1.1.csv.gz',
     'GPL13534_HumanMethylation450_15017482_v.1.1.csv'),
    # 4. GEO download page
    ('https://www.ncbi.nlm.nih.gov/geo/download/?acc=GPL13534&format=file'
     '&file=GPL13534_HumanMethylation450_15017482_v.1.1.csv.gz',
     'GPL13534_HumanMethylation450_15017482_v.1.1.csv'),
]

import gzip as _gz, shutil as _sh, subprocess as _sp

def try_load_manifest(path):
    """Try to load manifest from path. Returns df or None."""
    if not path or not os.path.exists(path): return None
    sz = os.path.getsize(path)/1e6
    if sz < 1: return None
    USECOLS = ['IlmnID','Name','CHR','MAPINFO','Strand','CpG_chrm','CpG_beg']
    for skip in [0, 7, 1, 8]:
        try:
            df = pd.read_csv(path, skiprows=skip, low_memory=False,
                             usecols=lambda c: c in USECOLS)
            # Accept if any known probe-ID or position column is present
            if not any(c in df.columns for c in ['Name','IlmnID','CHR','CpG_chrm']):
                continue
            # Fix duplicate columns immediately on load
            if df.columns.duplicated().any():
                df = df.loc[:, ~df.columns.duplicated()]
            return df
        except Exception:
            continue
    return None

# ── Step 1: try MANIFEST_PATH from config ────────────────────────────────────
manifest = try_load_manifest(MANIFEST_PATH)
if manifest is not None:
    print(f'  Loaded from config path: {MANIFEST_PATH}')

# ── Step 2: search Drive for any existing manifest ───────────────────────────
if manifest is None:
    import glob as _glm
    SEARCH_DIRS = [
        DATASET_BASE, PROJECT_DIR,
        '/content/drive/MyDrive/Meth3DNet_V6',
        '/content/drive/MyDrive',
    ]
    MANIFEST_PATTERNS = [
        '*HumanMethylation450*v.1.1.csv',
        '*HumanMethylation450*v.1.2.csv',
        '*HumanMethylation450k*v3.csv',
        'HM450*manifest*.tsv',
        'GPL13534*.csv',
    ]
    for sdir in dict.fromkeys(SEARCH_DIRS):
        if not os.path.isdir(sdir): continue
        for pat in MANIFEST_PATTERNS:
            hits = _glm.glob(os.path.join(sdir, '**', pat), recursive=True)
            for h in hits:
                if os.path.getsize(h) > 5_000_000:   # must be >5 MB
                    manifest = try_load_manifest(h)
                    if manifest is not None:
                        MANIFEST_PATH = h
                        print(f'  Found on Drive: {h}')
                        break
            if manifest is not None: break
        if manifest is not None: break

# ── Step 3: auto-download from known working URLs ────────────────────────────
if manifest is None:
    print('  Manifest not found on Drive. Downloading automatically...')
    manifest_dest = os.path.join(DATASET_BASE,
                    'GPL13534_HumanMethylation450_15017482_v.1.1.csv')
    os.makedirs(DATASET_BASE, exist_ok=True)

    for url, fname in MANIFEST_URLS_NB09:
        gz_tmp  = f'/tmp/manifest_dl.gz'
        out_tmp = f'/tmp/{fname}'
        print(f'  Trying: {url[:75]}...')
        r = _sp.run(['wget','--quiet','--tries=2','--timeout=120',
                     '-O', gz_tmp, url], capture_output=True)
        if not (r.returncode==0 and os.path.exists(gz_tmp)
                and os.path.getsize(gz_tmp)>100_000):
            r2 = _sp.run(['curl','--silent','--retry','2','--max-time','120',
                          '--location','-o',gz_tmp, url], capture_output=True)
            if not (r2.returncode==0 and os.path.exists(gz_tmp)
                    and os.path.getsize(gz_tmp)>100_000):
                if os.path.exists(gz_tmp): os.remove(gz_tmp)
                print('    FAILED'); continue
        try:
            with _gz.open(gz_tmp,'rb') as fi:
                with open(out_tmp,'wb') as fo:
                    _sh.copyfileobj(fi, fo, length=16*1024*1024)
            os.remove(gz_tmp)
        except Exception:
            _sh.move(gz_tmp, out_tmp)
        if os.path.exists(out_tmp) and os.path.getsize(out_tmp)>5_000_000:
            _sh.copy2(out_tmp, manifest_dest)
            manifest = try_load_manifest(manifest_dest)
            if manifest is not None:
                MANIFEST_PATH = manifest_dest
                sz = os.path.getsize(manifest_dest)/1e6
                print(f'    Downloaded and saved to Drive: {sz:.0f} MB')
                break
        print(f'    File too small or unreadable, trying next URL')

# ── Step 4: methylprep fallback ───────────────────────────────────────────────
if manifest is None:
    print('  Trying methylprep auto-download...')
    try:
        import subprocess
        subprocess.run(['pip','install','-q','methylprep'], check=True)
        from methylprep.files.manifests import Manifest
        from methylprep.models import ArrayType
        m_obj = Manifest(ArrayType.ILLUMINA_450K)
        manifest = m_obj.data_frame.reset_index()
        print(f'  methylprep manifest loaded: {len(manifest):,} probes')
    except Exception as e:
        print(f'  methylprep failed: {e}')

if manifest is None:
    raise RuntimeError(
        'Cannot load manifest. Manual steps:\n'
        '  1. Go to: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GPL13534\n'
        '  2. Download the CSV file (Supplementary files section)\n'
        '  3. Gunzip it, upload to: '
        + os.path.join(DATASET_BASE, 'GPL13534_HumanMethylation450_15017482_v.1.1.csv')
    )

# ── Standardise manifest ──────────────────────────────────────────────────────
# Rename probe ID column — only ONE of IlmnID/Name should become ProbeID
# Priority: IlmnID > Name (IlmnID is the canonical Illumina identifier)
if 'IlmnID' in manifest.columns:
    manifest.rename(columns={'IlmnID':'ProbeID'}, inplace=True)
    if 'Name' in manifest.columns:    # drop redundant Name column
        manifest.drop(columns=['Name'], inplace=True)
elif 'Name' in manifest.columns:
    manifest.rename(columns={'Name':'ProbeID'}, inplace=True)
# Safety: if somehow ProbeID is still duplicated, keep first occurrence
if manifest.columns.duplicated().any():
    manifest = manifest.loc[:, ~manifest.columns.duplicated()]
# Some methylprep manifests use 'CpG_chrm' instead of 'CHR'
if 'CHR' not in manifest.columns:
    for alt in ['CpG_chrm','chr','chrom','Chromosome']:
        if alt in manifest.columns:
            manifest.rename(columns={alt:'CHR'}, inplace=True)
            break
# v3 manifest uses 'MAPINFO' directly; older versions use 'CpG_beg'
if 'MAPINFO' not in manifest.columns:
    for alt in ['CpG_beg','MAPINFO','pos','Position','Start','MapInfo']:
        if alt in manifest.columns:
            manifest.rename(columns={alt:'MAPINFO'}, inplace=True)
            break

manifest = manifest.dropna(subset=['CHR'])
manifest['CHR'] = manifest['CHR'].astype(str).str.replace('chr','').str.strip()
manifest = manifest[~manifest['CHR'].str.contains(
    r'\*|random|Un', na=True, regex=True)]
if 'MAPINFO' in manifest.columns:
    manifest['MAPINFO'] = pd.to_numeric(manifest['MAPINFO'], errors='coerce')

print(f'Manifest ready: {len(manifest):,} probes')
print(f'  Chromosomes: {sorted(manifest["CHR"].unique(), key=chr_key)[:6]} ...')
print(f'  Columns: {[c for c in manifest.columns if c in ["ProbeID","CHR","MAPINFO","Strand"]]}')


Loading Illumina 450k manifest...
  Loaded from config path: /content/drive/MyDrive/Meth3DNet_V6/MultiCancer/GPL13534_HumanMethylation450_15017482_v.1.1.csv
Manifest ready: 485,547 probes
  Chromosomes: ['1', '2', '3', '4', '5', '6'] ...
  Columns: ['ProbeID', 'CHR', 'MAPINFO', 'Strand']


## Cell 6 — Core Functions

In [7]:
TUMOUR_WORDS = ['tumor','tumour','cancer','carcinoma','adenocarcinoma','malignant',
                'glioblastoma','gbm','hcc','hepatocell','squamous','adenoma',
                'primary','grade','stage','luad','lusc','brca','nsclc']
NORMAL_WORDS = ['normal','adjacent','control','healthy','non-tumor','non-tumour',
                'benign','nontumor','peritumoral','uninvolved','mucosa']

def parse_geo_matrix(filepath):
    with open(filepath,'r',encoding='utf-8',errors='replace') as fh:
        first = fh.readline()
    if first.startswith('!') or '!Series_title' in first or '!series_matrix' in first:
        return _parse_series_matrix(filepath)
    return _parse_tab_beta(filepath)

def _parse_series_matrix(filepath):
    sample_ids=[]; source_map={}; in_table=False; rows=[]
    with open(filepath,'r',encoding='utf-8',errors='replace') as fh:
        for line in fh:
            line = line.rstrip('\n')
            if '!Sample_geo_accession' in line:
                sample_ids=[p.strip('"') for p in line.split('\t')[1:]]
            for tag in ['!Sample_source_name_ch1','!Sample_characteristics_ch1','!Sample_title']:
                if tag in line and sample_ids:
                    parts=[p.strip('"').lower() for p in line.split('\t')[1:]]
                    for sid,src in zip(sample_ids,parts):
                        source_map[sid]=source_map.get(sid,'')+' '+src
            if '!series_matrix_table_begin' in line: in_table=True; continue
            if '!series_matrix_table_end' in line: break
            if in_table: rows.append(line)
    if not rows: raise ValueError(f'No table data in {filepath}')
    beta=pd.read_csv(StringIO('\n'.join(rows)),sep='\t',index_col=0)
    beta.index.name='ProbeID'
    beta=beta.apply(pd.to_numeric,errors='coerce')
    beta=beta[beta.isna().mean(axis=1)<=0.20]
    tumour_ids,normal_ids=[],[]
    for sid in sample_ids:
        if sid not in beta.columns: continue
        src=source_map.get(sid,'').lower()
        is_n=any(w in src for w in NORMAL_WORDS)
        is_t=any(w in src for w in TUMOUR_WORDS)
        if is_n and not is_t: normal_ids.append(sid)
        else: tumour_ids.append(sid)
    return beta, tumour_ids, normal_ids

def _parse_tab_beta(filepath):
    print('  Tab-separated beta format detected (e.g. GSE56044_methylation_raw.txt)')
    beta=pd.read_csv(filepath,sep='\t',index_col=0,low_memory=False)
    beta.index.name='ProbeID'
    beta=beta.apply(pd.to_numeric,errors='coerce')
    beta=beta[beta.isna().mean(axis=1)<=0.20]
    return beta, list(beta.columns), []

def map_probes(v6_df, manifest_df, tol=1000):
    # Guard: remove any duplicate columns from either df before merging
    if manifest_df.columns.duplicated().any():
        manifest_df = manifest_df.loc[:, ~manifest_df.columns.duplicated()]
    if v6_df.columns.duplicated().any():
        v6_df = v6_df.loc[:, ~v6_df.columns.duplicated()]
    if 'ProbeID' in v6_df.columns:
        return v6_df.merge(manifest_df[['ProbeID','CHR','MAPINFO']].dropna(),
                           on='ProbeID',how='inner')
    elif 'CHR' in v6_df.columns and 'START' in v6_df.columns:
        man=manifest_df[['ProbeID','CHR','MAPINFO']].dropna().copy()
        man['CHR']=man['CHR'].astype(str).str.replace('chr','')
        man['MAPINFO']=man['MAPINFO'].astype(int)
        man=man.sort_values(['CHR','MAPINFO'])
        dmb=v6_df.copy()
        dmb['CHR']=dmb['CHR'].astype(str).str.replace('chr','')
        dmb['START']=dmb['START'].astype(int)
        dmb=dmb.sort_values(['CHR','START'])
        res=[]
        for chrom in dmb['CHR'].unique():
            d=dmb[dmb['CHR']==chrom]; m=man[man['CHR']==chrom]
            if len(m)==0: continue
            mc=pd.merge_asof(d,m,left_on='START',right_on='MAPINFO',
                             tolerance=tol,direction='nearest')
            res.append(mc.dropna(subset=['ProbeID']))
        return pd.concat(res,ignore_index=True) if res else pd.DataFrame()
    raise ValueError(f'V6 needs ProbeID or CHR+START. Cols: {list(v6_df.columns)}')

def three_layers(v6p, beta, tc, nc, key, lc_possible=True):
    res={'key':key}
    tc=[c for c in tc if c in beta.columns] or list(beta.columns)
    cv=beta[tc].var(axis=1,ddof=1).rename('cv')
    vdf=v6p.set_index('ProbeID').join(cv,how='inner').dropna(subset=['cv'])
    res['n_matched']=len(vdf)
    hi=vdf.loc[vdf['dmb_group']=='High','cv']
    lo=vdf.loc[vdf['dmb_group']=='Low','cv']
    mw_p=(mannwhitneyu(hi,lo,alternative='greater')[1]
          if len(hi)>0 and len(lo)>0 else np.nan)
    fold=hi.median()/lo.median() if lo.median()>0 else np.nan
    r_a,p_a=spearmanr(vdf['abs_delta'],vdf['cv'])
    res['A']={'fold':fold,'mw_p':mw_p,'r':r_a,'p':p_a,
              'n_hi':len(hi),'n_lo':len(lo),'hi':hi.values,'lo':lo.values,
              'med_hi':hi.median(),'med_lo':lo.median()}
    res['B']=None
    if 'ct_z' in vdf.columns:
        ct=vdf[['ct_z','cv']].dropna()
        r_b,p_b=spearmanr(ct['ct_z'],ct['cv'])
        hi_ct=ct[ct['ct_z']>2.33]['cv']; lo_ct=ct[ct['ct_z']<=2.33]['cv']
        ct_fold=(hi_ct.median()/lo_ct.median()
                 if len(hi_ct)>5 and lo_ct.median()>0 else np.nan)
        res['B']={'r':r_b,'p':p_b,'n':len(ct),'ct_fold':ct_fold}
    res['C']=None
    nc2=[c for c in nc if c in beta.columns]
    if lc_possible and len(tc)>0 and len(nc2)>=3 and 'direction' in vdf.columns:
        cd=(beta[tc].mean(axis=1)-beta[nc2].mean(axis=1)).rename('cd')
        vc=vdf.join(cd,how='inner').dropna(subset=['cd','direction'])
        # Cast direction to numeric (may be string '1'/'-1' from CSV)
        vc=vc.copy()
        vc['direction']=pd.to_numeric(vc['direction'],errors='coerce')
        vc=vc.dropna(subset=['direction'])
        r_c,p_c=spearmanr(vc['direction'],vc['cd'])
        conc=(np.sign(vc['direction'].astype(float))==np.sign(vc['cd'])).sum()
        res['C']={'r':r_c,'p':p_c,'conc_pct':100*conc/len(vc),
                  'n':len(vc),'n_t':len(tc),'n_n':len(nc2)}
    elif not lc_possible:
        res['C']={'note':'Tumour-only dataset — Layer C not applicable'}
    return res

def per_chr(v6p, beta, tc):
    tc=[c for c in tc if c in beta.columns] or list(beta.columns)
    cv=beta[tc].var(axis=1,ddof=1)
    vdf=v6p.set_index('ProbeID').join(cv.rename('v'),how='inner').dropna(subset=['v'])
    cc=next((c for c in ['CHR','CHR_x','chr'] if c in vdf.columns),None)
    if not cc: return pd.DataFrame()
    rows=[]
    for chrom,grp in vdf.groupby(cc):
        if len(grp)<20: continue
        hi=grp[grp['dmb_group']=='High']['v']; lo=grp[grp['dmb_group']=='Low']['v']
        fold=hi.median()/lo.median() if (len(lo)>0 and lo.median()>0) else np.nan
        r_c,p_c=spearmanr(grp['abs_delta'],grp['v']) if len(grp)>10 else (np.nan,np.nan)
        rows.append({'CHR':str(chrom),'n':len(grp),'n_hi':len(hi),'n_lo':len(lo),
                     'fold':fold,'r':r_c,'p':p_c})
    return pd.DataFrame(rows).sort_values('CHR',key=lambda x:x.map(chr_key))

print('Core functions ready.')

Core functions ready.


## Cell 7 — Map V6 Probes to 450k Array

In [8]:
print('Mapping V6 DMBs to 450k probe IDs...')
v6_probes = map_probes(V6, manifest)
if len(v6_probes)==0:
    raise ValueError('Zero probe mappings. Check V6 column names and manifest.')
v6_probes=(v6_probes.sort_values('abs_delta',ascending=False)
                    .drop_duplicates('ProbeID').reset_index(drop=True))
print(f'Mapped: {len(v6_probes):,} unique probes')
print(f'  High: {(v6_probes.dmb_group=="High").sum():,}  '
      f'Low: {(v6_probes.dmb_group=="Low").sum():,}')

Mapping V6 DMBs to 450k probe IDs...
Mapped: 41,217 unique probes
  High: 6,343  Low: 34,874


## Cell 8 — Main Validation Loop (all 7 datasets)
> ⏱ Runtime: ~90–150 min for all 7 datasets on Colab GPU/High-RAM. Individual cancer datasets take 5–25 min each.

In [9]:
ALL_RESULTS = {}
CHR_STATS   = {}
SKIPPED     = []
RUNTIMES    = {}

ORDER = [
    'Lung_GSE39279','Lung_GSE56044',
    'Breast_GSE75067',
    'CRC_GSE101764','CRC_GSE48684',
    'HCC_GSE54503','GBM_GSE36278'
]

total_start = time.time()

for key in ORDER:
    d   = DATASETS[key]
    mat = MATRIX_PATHS.get(key)
    t0  = time.time()

    print(f'\n{"="*65}')
    print(f'[{d["group"]}]  {d["cancer"]}  ({d["gse"]})')
    print(f'{"="*65}')

    if mat is None or not os.path.exists(mat):
        print(f'  SKIPPED — file not found')
        print(f'  Upload to Drive: {DATASET_BASE}/{d["subdir"]}/')
        print(f'  Download: https://ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={d["gse"]}')
        SKIPPED.append(key); continue

    # ── Load beta matrix ───────────────────────────────────────────────────────
    try:
        beta, tc, nc = parse_geo_matrix(mat)
        print(f'  Beta: {beta.shape[0]:,} probes x {beta.shape[1]} samples  '
              f'[Tumour={len(tc)}  Normal={len(nc)}]')
        if len(tc)==0:
            print('  Auto-classify failed — treating all samples as tumour')
            tc = list(beta.columns)
    except Exception as e:
        import traceback; traceback.print_exc()
        SKIPPED.append(key); continue

    # ── Three-layer validation ─────────────────────────────────────────────────
    try:
        res = three_layers(v6_probes, beta, tc, nc, key,
                           lc_possible=d.get('layer_c_possible',True))
        res['meta'] = d
        A = res['A']
        print(f'  Layer A: fold={A["fold"]:.3f}x  {fmt_p(A["mw_p"])}')
        print(f'           Spearman r={A["r"]:.4f}  {fmt_p(A["p"])}')
        if res['B']:
            print(f'  Layer B: CT r={res["B"]["r"]:.4f}  {fmt_p(res["B"]["p"])}')
        C = res.get('C')
        if C and C.get('conc_pct') is not None:
            print(f'  Layer C: {C["conc_pct"]:.1f}% concordant  '
                  f'r={C["r"]:.4f}  {fmt_p(C["p"])}')
        elif C and 'note' in C:
            print(f'  Layer C: {C["note"]}')
        ALL_RESULTS[key] = res
    except Exception as e:
        import traceback; traceback.print_exc()
        SKIPPED.append(key); del beta; gc.collect(); continue

    # ── Per-chromosome stats ───────────────────────────────────────────────────
    try:
        cdf = per_chr(v6_probes, beta, tc)
        CHR_STATS[key] = cdf
        cdf.to_csv(f'{OUT_DIR}/{key}_perChr.csv', index=False)
    except Exception as e:
        print(f'  per-chr error: {e}')

    elapsed = time.time()-t0
    RUNTIMES[key] = elapsed
    print(f'  Runtime: {elapsed/60:.1f} min')
    del beta; gc.collect()

total_elapsed = time.time()-total_start
print(f'\n{"="*65}')
print(f'COMPLETE: {len(ALL_RESULTS)} datasets  |  Skipped: {len(SKIPPED)} {SKIPPED}')
print(f'Total runtime: {total_elapsed/60:.1f} min')
for k,t in RUNTIMES.items():
    print(f'  {k:<25} {t/60:.1f} min')


[Lung]  Lung adenocarcinoma (LUAD)  (GSE39279)
  Beta: 485,564 probes x 444 samples  [Tumour=444  Normal=0]
  Layer A: fold=1.352x  p=2.8e-91
           Spearman r=0.1345  p=1.4e-165
  Layer B: CT r=0.0666  p=8.5e-42
  Runtime: 1.2 min

[Lung]  Lung cancer multi-hist. (LUAD+LUSC)  (GSE56044)
  Beta: 485,145 probes x 136 samples  [Tumour=124  Normal=12]
  Layer A: fold=1.407x  p=1.3e-89
           Spearman r=0.1288  p=1.2e-151
  Layer B: CT r=0.0657  p=1.5e-40
  Layer C: 58.0% concordant  r=0.0419  p=3.4e-11
  Runtime: 0.3 min

[Breast]  Breast cancer (BRCA)  (GSE75067)
  Beta: 484,759 probes x 188 samples  [Tumour=188  Normal=0]
  Layer A: fold=1.491x  p=9.2e-99
           Spearman r=0.1386  p=3.4e-175
  Layer B: CT r=0.0710  p=5.2e-47
  Layer C: Tumour-only dataset — Layer C not applicable
  Runtime: 0.7 min

[CRC]  Colorectal cancer (CRC)  (GSE101764)
  Beta: 485,511 probes x 261 samples  [Tumour=261  Normal=0]
  Layer A: fold=1.313x  p=2.4e-61
           Spearman r=0.1116  p=2.0e-1

## Cell 9 — Lung Cancer Meta-Analysis (GSE39279 + GSE56044)

In [10]:
lung_keys = [k for k in ['Lung_GSE39279','Lung_GSE56044'] if k in ALL_RESULTS]
LUNG_META = None

if len(lung_keys) == 0:
    print('No lung results available.')
elif len(lung_keys) == 1:
    print(f'Single lung cohort: {lung_keys[0]}')
    r = ALL_RESULTS[lung_keys[0]]
    LUNG_META = {'A_fold':r['A']['fold'],'A_mw_p':r['A']['mw_p'],'A_r':r['A']['r'],
                 'B_r':r['B']['r'] if r.get('B') else np.nan,
                 'B_p':r['B']['p'] if r.get('B') else np.nan,
                 'C_conc':r['C']['conc_pct'] if r.get('C') and r['C'].get('conc_pct') else np.nan,
                 'C_r':r['C']['r'] if r.get('C') and r['C'].get('r') else np.nan,
                 'n':DATASETS[lung_keys[0]]['n_approx'],
                 'hi':r['A']['hi'],'lo':r['A']['lo']}
else:
    hi_all = np.concatenate([ALL_RESULTS[k]['A']['hi'] for k in lung_keys])
    lo_all = np.concatenate([ALL_RESULTS[k]['A']['lo'] for k in lung_keys])
    mw_p   = mannwhitneyu(hi_all, lo_all, alternative='greater')[1]
    fold   = np.median(hi_all)/np.median(lo_all)
    r_mean = np.mean([ALL_RESULTS[k]['A']['r'] for k in lung_keys])
    b_rs   = [ALL_RESULTS[k]['B']['r'] for k in lung_keys if ALL_RESULTS[k].get('B')]
    b_ps   = [ALL_RESULTS[k]['B']['p'] for k in lung_keys if ALL_RESULTS[k].get('B')]
    c_pcts = [ALL_RESULTS[k]['C']['conc_pct'] for k in lung_keys
              if ALL_RESULTS[k].get('C') and ALL_RESULTS[k]['C'].get('conc_pct')]
    c_rs   = [ALL_RESULTS[k]['C']['r'] for k in lung_keys
              if ALL_RESULTS[k].get('C') and ALL_RESULTS[k]['C'].get('r')]
    n_tot  = sum(DATASETS[k]['n_approx'] for k in lung_keys)
    LUNG_META = {
        'A_fold':fold,'A_mw_p':mw_p,'A_r':r_mean,
        'B_r':np.mean(b_rs) if b_rs else np.nan,
        'B_p':np.min(b_ps) if b_ps else np.nan,
        'C_conc':np.mean(c_pcts) if c_pcts else np.nan,
        'C_r':np.mean(c_rs) if c_rs else np.nan,
        'n':n_tot,'hi':hi_all,'lo':lo_all
    }
    print(f'Lung meta-analysis ({" + ".join(lung_keys)}):')
    print(f'  Total n: ~{n_tot}')
    print(f'  Layer A: fold={fold:.3f}x  {fmt_p(mw_p)}')
    if c_pcts: print(f'  Layer C: {np.mean(c_pcts):.1f}% concordant')

Lung meta-analysis (Lung_GSE39279 + Lung_GSE56044):
  Total n: ~580
  Layer A: fold=1.351x  p=5.7e-148
  Layer C: 58.0% concordant


## Cell 10 — CRC Progression: Normal → Adenoma → Carcinoma

In [11]:
CRC_KEY = 'CRC_GSE48684'
PROG_DF = None
if CRC_KEY not in ALL_RESULTS:
    print('GSE48684 not loaded — skipping.')
else:
    mat = MATRIX_PATHS[CRC_KEY]
    sample_ids=[]; stage_map={}; in_table=False; rows=[]
    with open(mat,'r',encoding='utf-8',errors='replace') as fh:
        for line in fh:
            line=line.rstrip('\n')
            if '!Sample_geo_accession' in line:
                sample_ids=[p.strip('"') for p in line.split('\t')[1:]]
            for tag in ['!Sample_source_name_ch1','!Sample_characteristics_ch1','!Sample_title']:
                if tag in line and sample_ids:
                    parts=[p.strip('"').lower() for p in line.split('\t')[1:]]
                    for sid,src in zip(sample_ids,parts):
                        stage_map[sid]=stage_map.get(sid,'')+' '+src
            if '!series_matrix_table_begin' in line: in_table=True; continue
            if '!series_matrix_table_end' in line: break
            if in_table: rows.append(line)

    beta_crc=pd.read_csv(StringIO('\n'.join(rows)),sep='\t',index_col=0)
    beta_crc.index.name='ProbeID'
    beta_crc=beta_crc.apply(pd.to_numeric,errors='coerce')
    beta_crc=beta_crc[beta_crc.isna().mean(axis=1)<=0.20]

    norm_s,aden_s,canc_s=[],[],[]
    for sid,src in stage_map.items():
        if sid not in beta_crc.columns: continue
        if any(w in src for w in ['adenoma','polyp']): aden_s.append(sid)
        elif any(w in src for w in ['normal','adjacent','mucosa','control']): norm_s.append(sid)
        elif any(w in src for w in ['cancer','carcinoma','tumor']): canc_s.append(sid)

    print(f'CRC stages: Normal={len(norm_s)}  Adenoma={len(aden_s)}  Carcinoma={len(canc_s)}')
    prog=[]
    for stage,cols in [('Normal',norm_s),('Adenoma',aden_s),('Carcinoma',canc_s)]:
        cols=[c for c in cols if c in beta_crc.columns]
        if len(cols)<2: print(f'  Skip {stage}: {len(cols)} samples'); continue
        var=beta_crc[cols].var(axis=1,ddof=1)
        vdf=v6_probes.set_index('ProbeID').join(var.rename('v'),how='inner').dropna()
        hi=vdf[vdf['dmb_group']=='High']['v']; lo=vdf[vdf['dmb_group']=='Low']['v']
        fold=hi.median()/lo.median() if lo.median()>0 else np.nan
        mw_p=mannwhitneyu(hi,lo,alternative='greater')[1] if len(hi)>0 and len(lo)>0 else np.nan
        prog.append({'Stage':stage,'n':len(cols),'fold':fold,'mw_p':mw_p})
        print(f'  {stage:<12}: n={len(cols):3d}  fold={fold:.3f}x  {fmt_p(mw_p)}')

    PROG_DF=pd.DataFrame(prog)
    PROG_DF.to_csv(f'{OUT_DIR}/CRC_Progression.csv',index=False)

    if len(prog)>=2:
        fig_p,ax_p=plt.subplots(figsize=(6,4),facecolor='white')
        ax_p.plot(PROG_DF.Stage,PROG_DF.fold,'o-',color='#1B5E20',
                  lw=2.5,ms=12,mfc='white',mew=2.5)
        ax_p.fill_between(PROG_DF.Stage,1,PROG_DF.fold,alpha=0.12,color='#4CAF50')
        ax_p.axhline(1.0,color='#999',lw=1,ls='--')
        for _,r in PROG_DF.iterrows():
            ax_p.annotate(f'{r.fold:.3f}x',(r.Stage,r.fold),
                          textcoords='offset points',xytext=(0,12),
                          ha='center',fontsize=11,fontweight='bold')
        ax_p.set_ylabel('Variance enrichment fold',fontsize=10)
        ax_p.set_title('CRC Oncogenic Progression: V6 DMB Enrichment\n'
                       'Normal -> Adenoma -> Carcinoma (GSE48684)',
                       fontsize=11,fontweight='bold',color=NAVY)
        ax_p.grid(axis='y',alpha=0.2)
        plt.tight_layout()
        fig_p.savefig(f'{OUT_DIR}/Fig_CRC_Progression.png',dpi=150,bbox_inches='tight')
        fig_p.savefig(f'{OUT_DIR}/Fig_CRC_Progression.tif',dpi=300,bbox_inches='tight',format='tiff')
        plt.show(); print('CRC progression figure saved.')

    del beta_crc; gc.collect()

CRC stages: Normal=105  Adenoma=42  Carcinoma=0
  Normal      : n=105  fold=nanx  N/A
  Adenoma     : n= 42  fold=nanx  N/A
  Skip Carcinoma: 0 samples
CRC progression figure saved.


## Cell 11 — 7-Panel Validation Figure

In [12]:
if not ALL_RESULTS:
    print('No results — run Cell 8 first.')
else:
    ordered = [k for k in ORDER if k in ALL_RESULTS]
    n = len(ordered)
    ncols=4; nrows=int(np.ceil(n/ncols))
    fig,axes=plt.subplots(nrows,ncols,figsize=(4.4*ncols,4.6*nrows),facecolor='white')
    if n==1: axes=np.array([[axes]])
    elif nrows==1: axes=axes.reshape(1,-1)

    fig.suptitle('Meth3D-Net V6: Multi-Cancer + Lung Cancer Validation\n'
                 'Layer A — Variance Enrichment (High-|\u0394\u03b2| vs Low-|\u0394\u03b2|)',
                 fontsize=13,fontweight='bold',color=NAVY,y=1.01)

    for idx,key in enumerate(ordered):
        r,c=divmod(idx,ncols)
        ax=axes[r][c]
        A=ALL_RESULTS[key]['A']; m=ALL_RESULTS[key]['meta']
        clr=COLORS.get(key,'#888')
        vp=ax.violinplot([A['lo'],A['hi']],positions=[1,2],showmedians=True,showextrema=False)
        vp['bodies'][0].set_facecolor('#BBBBBB'); vp['bodies'][0].set_alpha(0.5)
        vp['bodies'][1].set_facecolor(clr);       vp['bodies'][1].set_alpha(0.78)
        vp['cmedians'].set_color('black'); vp['cmedians'].set_linewidth(2)

        title=m['cancer']
        for rm in ['(LUAD)','(BRCA)','(CRC)','(HCC)','(GBM)','multi-hist.',' (CRC)']:
            title=title.replace(rm,'').strip()
        ax.set_title(f'{title}\n{m["gse"]}  n~{m["n_approx"]}',
                     fontsize=9,fontweight='bold',color='#222',pad=3)
        ax.set_xticks([1,2])
        ax.set_xticklabels([f'Low\n(n={A["n_lo"]:,})',f'High\n(n={A["n_hi"]:,})'],fontsize=8)
        ax.set_ylabel('Methylation variance',fontsize=8)
        fold_s=f'{A["fold"]:.3f}x' if not np.isnan(A['fold']) else 'N/A'
        box_c='#FFF3E0' if m['group']=='Lung' else '#E8F5E9'
        ax.annotate(f'Fold={fold_s}\n{fmt_p(A["mw_p"])}',
                    xy=(0.97,0.97),xycoords='axes fraction',ha='right',va='top',fontsize=7.5,
                    bbox=dict(boxstyle='round,pad=0.3',facecolor=box_c,edgecolor='#CCC',alpha=0.9))
        ax.annotate(m['group'],xy=(0.03,0.97),xycoords='axes fraction',
                    ha='left',va='top',fontsize=7,fontweight='bold',
                    color=GROUP_COLORS.get(m['group'],'#333'))
        C=ALL_RESULTS[key].get('C')
        if C and C.get('conc_pct') is not None:
            ax.annotate(f"C: {C['conc_pct']:.1f}% (r={C['r']:.3f})",
                        xy=(0.03,0.04),xycoords='axes fraction',
                        ha='left',va='bottom',fontsize=7,color='#444')
        ax.grid(True,alpha=0.2)

    for idx in range(n,nrows*ncols):
        r,c=divmod(idx,ncols); axes[r][c].set_visible(False)

    plt.tight_layout()
    fig.savefig(f'{OUT_DIR}/Fig_AllCancer_Validation.png',dpi=150,bbox_inches='tight')
    fig.savefig(f'{OUT_DIR}/Fig_AllCancer_Validation.tif',dpi=300,bbox_inches='tight',format='tiff')
    plt.show()
    print(f'All-cancer figure saved ({n} panels).')

All-cancer figure saved (7 panels).


## Cell 12 — Fold Enrichment Summary Bar (all cohorts)

In [13]:
if not ALL_RESULTS:
    print('No results.')
else:
    bar=[
        {'label':'MB\n(GSE85212)','fold':1.62,'p':2.2e-300,'color':'#1F4E79','g':'Reference'},
        {'label':'TCGA\n(33 types)','fold':1.36,'p':0.001,'color':'#5B8DB8','g':'Reference'},
    ]
    if LUNG_META and not np.isnan(LUNG_META.get('A_fold',np.nan)):
        bar.append({'label':'Lung\n(GSE39279\n+GSE56044)',
                    'fold':LUNG_META['A_fold'],'p':LUNG_META['A_mw_p'],
                    'color':'#0277BD','g':'Lung'})
    non_lung=[k for k in ORDER if k in ALL_RESULTS and DATASETS[k]['group']!='Lung']
    for key in sorted(non_lung,key=lambda k:ALL_RESULTS[k]['A']['fold']
                      if not np.isnan(ALL_RESULTS[k]['A']['fold']) else 0,reverse=True):
        m=ALL_RESULTS[key]['meta']; A=ALL_RESULTS[key]['A']
        bar.append({'label':f'{m["group"]}\n({m["gse"]})',
                    'fold':A['fold'] if not np.isnan(A['fold']) else 0,
                    'p':A['mw_p'],'color':COLORS.get(key,'#888'),'g':m['group']})

    fig2,ax2=plt.subplots(figsize=(max(10,len(bar)*1.5),5.5),facecolor='white')
    folds=[d['fold'] if not np.isnan(d['fold']) else 0 for d in bar]
    ax2.bar(range(len(bar)),folds,color=[d['color'] for d in bar],
            edgecolor='white',linewidth=0.5,alpha=0.88,zorder=3)
    ax2.axhline(1.0,color='#333',lw=1.3,ls='--',label='No enrichment',zorder=2)
    for i,d in enumerate(bar):
        p=d['p']
        star='***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns' if not np.isnan(p) else ''
        if folds[i]>0:
            ax2.text(i,folds[i]+0.006,star,ha='center',va='bottom',fontsize=10,fontweight='bold')
            ax2.text(i,folds[i]/2,f'{folds[i]:.3f}x',ha='center',va='center',
                     fontsize=8,color='white',fontweight='bold')
    legend_items=(
        [Patch(facecolor='#1F4E79',label='MB'),Patch(facecolor='#5B8DB8',label='TCGA')] +
        [Patch(facecolor=c,label=g) for g,c in GROUP_COLORS.items()]
    )
    ax2.legend(handles=legend_items,fontsize=8,loc='upper right',framealpha=0.85)
    ax2.set_xticks(range(len(bar)))
    ax2.set_xticklabels([d['label'] for d in bar],fontsize=8.5,rotation=25,ha='right')
    ax2.set_ylabel('Variance enrichment fold',fontsize=10)
    ax2.set_title('Meth3D-Net V6: Layer A Enrichment Across All Cancer Types\n'
                  '(*** p<0.001; ** p<0.01; * p<0.05)',
                  fontsize=12,fontweight='bold',color=NAVY)
    ax2.grid(axis='y',alpha=0.2,zorder=1)
    plt.tight_layout()
    fig2.savefig(f'{OUT_DIR}/Fig_FoldEnrichment_AllCancers.png',dpi=150,bbox_inches='tight')
    fig2.savefig(f'{OUT_DIR}/Fig_FoldEnrichment_AllCancers.tif',dpi=300,bbox_inches='tight',format='tiff')
    plt.show()
    print('Fold enrichment bar saved.')

Fold enrichment bar saved.


## Cell 13 — Supplementary Table S_MultiCancer

In [14]:
rows=[]
REFS=[
    {'l':'Medulloblastoma (MB)','g':'GSE85212','n':763,
     'af':'1.62x','ap':'<2.2e-300','ar':'0.039',
     'br':'0.039','bp':'5.9e-30','cp':'58.4%','cr':'0.025','nb':'NB01'},
    {'l':'TCGA Pan-Cancer','g':'PanCanAtlas','n':9854,
     'af':'1.36x','ap':'<0.001','ar':'0.271',
     'br':'0.013','bp':'4.1e-5','cp':'40.8%','cr':'-0.248','nb':'NB05'},
]
for s in REFS:
    rows.append({'Cancer':s['l'],'GEO':s['g'],'n':s['n'],
                 'A fold':s['af'],'A p':s['ap'],'A r':s['ar'],
                 'B CT r':s['br'],'B CT p':s['bp'],
                 'C %':s['cp'],'C r':s['cr'],'Source':s['nb']})

if LUNG_META and not np.isnan(LUNG_META.get('A_fold',np.nan)):
    lm=LUNG_META
    rows.append({'Cancer':'Lung cancer (meta: GSE39279+GSE56044)',
                 'GEO':'GSE39279+GSE56044','n':lm.get('n','~365'),
                 'A fold':f"{lm['A_fold']:.3f}x",'A p':fmt_p(lm['A_mw_p']),
                 'A r':f"{lm['A_r']:.4f}" if not np.isnan(lm['A_r']) else 'N/A',
                 'B CT r':f"{lm['B_r']:.4f}" if not np.isnan(lm.get('B_r',np.nan)) else 'N/A',
                 'B CT p':fmt_p(lm.get('B_p')),
                 'C %':f"{lm['C_conc']:.1f}%" if lm.get('C_conc') and not np.isnan(lm['C_conc']) else 'N/A',
                 'C r':f"{lm['C_r']:.4f}" if lm.get('C_r') and not np.isnan(lm['C_r']) else 'N/A',
                 'Source':'NB09-Colab (Cell 9)'})

for key in ORDER:
    if key not in ALL_RESULTS: continue
    res=ALL_RESULTS[key]; m=res['meta']
    A=res['A']; B=res.get('B') or {}; C=res.get('C') or {}
    cp=(f"{C['conc_pct']:.1f}%" if C.get('conc_pct') is not None
        else C.get('note','N/A (tumour-only)'))
    rows.append({'Cancer':m['cancer'],'GEO':m['gse'],'n':m['n_approx'],
                 'A fold':f"{A['fold']:.3f}x" if not np.isnan(A['fold']) else 'N/A',
                 'A p':fmt_p(A['mw_p']),
                 'A r':f"{A['r']:.4f}" if not np.isnan(A['r']) else 'N/A',
                 'B CT r':f"{B['r']:.4f}" if B.get('r') is not None else 'N/A',
                 'B CT p':fmt_p(B.get('p')),
                 'C %':cp,
                 'C r':f"{C['r']:.4f}" if C.get('r') is not None else 'N/A',
                 'Source':'NB09-Colab (Cell 8)'})

df=pd.DataFrame(rows)
df.to_csv(f'{OUT_DIR}/SupTable_S_MultiCancer.csv',index=False)
print(f'Saved: SupTable_S_MultiCancer.csv  ({len(df)} rows)')
print(df[['Cancer','A fold','A p','C %']].to_string(index=False))

Saved: SupTable_S_MultiCancer.csv  (10 rows)
                               Cancer A fold        A p                                          C %
                 Medulloblastoma (MB)  1.62x  <2.2e-300                                        58.4%
                      TCGA Pan-Cancer  1.36x     <0.001                                        40.8%
Lung cancer (meta: GSE39279+GSE56044) 1.351x p=5.7e-148                                        58.0%
           Lung adenocarcinoma (LUAD) 1.352x  p=2.8e-91                            N/A (tumour-only)
  Lung cancer multi-hist. (LUAD+LUSC) 1.407x  p=1.3e-89                                        58.0%
                 Breast cancer (BRCA) 1.491x  p=9.2e-99 Tumour-only dataset — Layer C not applicable
              Colorectal cancer (CRC) 1.313x  p=2.4e-61                            N/A (tumour-only)
         Colorectal progression (CRC) 1.550x  p=4.0e-46                                        63.3%
       Hepatocellular carcinoma (HCC) 1.690x p

## Cell 14 — Save Outputs to Google Drive & Final Summary

In [15]:
import shutil

# ── Copy outputs to Google Drive for persistence ──────────────────────────────
if IN_COLAB:
    drive_out = os.path.join(DRIVE_BASE, 'Meth3DNet_V6', 'NB09_outputs')
    os.makedirs(drive_out, exist_ok=True)
    copied = []
    for f in os.listdir(OUT_DIR):
        src = os.path.join(OUT_DIR, f)
        dst = os.path.join(drive_out, f)
        shutil.copy2(src, dst)
        copied.append(f)
    print(f'Copied {len(copied)} files to Google Drive: {drive_out}')
    for f in copied:
        print(f'  {f}')
    print()

# ── Final summary ─────────────────────────────────────────────────────────────
print('='*68)
print('NB09 Colab — COMPLETE')
print('='*68)
print(f'Datasets: {len(ALL_RESULTS)} completed  |  {len(SKIPPED)} skipped {SKIPPED}')
print(f'Output dir: {OUT_DIR}')
for f in sorted(os.listdir(OUT_DIR)):
    mb=os.path.getsize(os.path.join(OUT_DIR,f))/1e6
    print(f'  {f:<55} {mb:6.2f} MB')

if ALL_RESULTS:
    print('\nLayer A fold enrichment (all cancers):')
    if LUNG_META and not np.isnan(LUNG_META.get('A_fold',np.nan)):
        print(f'  Lung meta-analysis               fold={LUNG_META["A_fold"]:.3f}x  {fmt_p(LUNG_META["A_mw_p"])}')
    for key in sorted(ALL_RESULTS,
                      key=lambda k: ALL_RESULTS[k]['A']['fold']
                      if not np.isnan(ALL_RESULTS[k]['A']['fold']) else 0, reverse=True):
        m=ALL_RESULTS[key]['meta']; A=ALL_RESULTS[key]['A']
        print(f'  {m["gse"]:<15} {m["cancer"]:<42} fold={A["fold"]:.3f}x  {fmt_p(A["mw_p"])}')

print('\nCitations to add:')
seen=set()
for key in ALL_RESULTS:
    c=ALL_RESULTS[key]['meta']['citation']
    if c not in seen: print(f'  {c}'); seen.add(c)

if LUNG_META:
    print('  Selamat SA et al. (2012) Genome Res 22:1197-1211. [GSE39279]')
    print('  Sandoval J et al. (2013) Epigenetics 6:692-702. [GSE56044]')

print('\nNext steps for paper:')
print('  1. SupTable_S_MultiCancer.csv      -> Supplementary Table S_MultiCancer')
print('  2. Fig_AllCancer_Validation.tif    -> Supplementary Figure S7')
print('  3. Fig_FoldEnrichment_AllCancers.tif -> Figure 4 new panel')
print('  4. Fig_CRC_Progression.tif         -> Supplementary Figure S8')
print('  5. Results text template -> Cell 13 of Kaggle NB09')
print()
print('Repository: https://github.com/neetuaashi/Meth3D-Net')
print('Archive:    https://zenodo.org/records/19657976')

Copied 16 files to Google Drive: /content/drive/MyDrive/Meth3DNet_V6/NB09_outputs
  HCC_GSE54503_perChr.csv
  GBM_GSE36278_perChr.csv
  Fig_FoldEnrichment_AllCancers.png
  CRC_GSE48684_perChr.csv
  Fig_FoldEnrichment_AllCancers.tif
  Fig_AllCancer_Validation.png
  Lung_GSE39279_perChr.csv
  Breast_GSE75067_perChr.csv
  Fig_CRC_Progression.tif
  Fig_AllCancer_Validation.tif
  CRC_GSE101764_perChr.csv
  CRC_Progression.csv
  SupTable_S_MultiCancer.csv
  Lung_GSE56044_perChr.csv
  DRIVE_SETUP_GUIDE.txt
  Fig_CRC_Progression.png

NB09 Colab — COMPLETE
Datasets: 7 completed  |  0 skipped []
Output dir: /content/multicancer_output
  Breast_GSE75067_perChr.csv                                0.00 MB
  CRC_GSE101764_perChr.csv                                  0.00 MB
  CRC_GSE48684_perChr.csv                                   0.00 MB
  CRC_Progression.csv                                       0.00 MB
  DRIVE_SETUP_GUIDE.txt                                     0.00 MB
  Fig_AllCancer_Validation.